In [4]:
import os
import sys
import warnings
import pandas as pd
import numpy as np
from scipy import stats
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score)
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier

if hasattr(sys.stdout, 'reconfigure'):
    try:
        sys.stdout.reconfigure(encoding='utf-8')
    except Exception:
        pass

warnings.filterwarnings('ignore')

print('======================================================================')
print('STEP 6.5: DATA SPLIT OPTIMIZATION EXPERIMENT & ABLATION STUDY')
print('======================================================================')

df = pd.read_csv('data_with_cost_matrix.csv')
print(f'Dataset shape: {df.shape[0]:,} rows x {df.shape[1]} columns')

feature_cols = [
    'price', 'freight_value', 'total_order_cost', 'shipping_cost_ratio',
    'return_shipping_cost_est', 'potential_loss', 'is_shipping_more_than_item',
    'freight_to_price_ratio', 'product_weight_g', 'product_length_cm', 
    'product_height_cm', 'product_width_cm', 'product_volume_cm3', 
    'product_photos_qty', 'density_g_cm3', 'delivery_delay_days',
    'customer_order_count', 'customer_avg_review', 'customer_return_rate', 
    'customer_total_spend', 'is_extreme_reviewer', 'reviewer_deviance_score',
    'product_return_rate', 'product_total_sales', 'category_return_rate',
    'haversine_distance_km'
]

le = LabelEncoder()
df['category_encoded'] = le.fit_transform(df['product_category_name_english'].fillna('unknown'))
feature_cols.append('category_encoded')
feature_cols = [c for c in feature_cols if c in df.columns]

X = df[feature_cols]
y = df['is_returned']
sample_weights = df['sample_cost_weight'] if 'sample_cost_weight' in df.columns else None

def compute_loss(y_true, y_prob, thresh, indices):
    y_pred = (y_prob >= thresh).astype(int)
    fn_mask = (y_true == 1) & (y_pred == 0)
    fp_mask = (y_true == 0) & (y_pred == 1)
    fn_cost = df.loc[indices[fn_mask], 'cost_FN'].sum()
    fp_cost = df.loc[indices[fp_mask], 'cost_FP'].sum()
    return fn_cost + fp_cost, fn_cost, fp_cost, y_pred

results = []
thresholds = np.arange(0.05, 0.95, 0.01)

# --- Method 1: 80/20 Direct Test Search ---
X_tr80, X_te20, y_tr80, y_te20, w_tr80, w_te20 = train_test_split(X, y, sample_weights, test_size=0.2, random_state=42, shuffle=False)
spw1 = (y_tr80 == 0).sum() / (y_tr80 == 1).sum()
xgb1 = XGBClassifier(n_estimators=300, max_depth=8, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, scale_pos_weight=spw1, reg_alpha=0.1, reg_lambda=1.0, random_state=42, eval_metric='logloss')
xgb1.fit(X_tr80, y_tr80, sample_weight=w_tr80)
p1 = xgb1.predict_proba(X_te20)[:, 1]
idx1 = X_te20.index
b_t1, m_l1 = 0.5, float('inf')
for t in thresholds:
    l, _, _, _ = compute_loss(y_te20, p1, t, idx1)
    if l < m_l1: m_l1, b_t1 = l, t
l1, fn1, fp1, pred1 = compute_loss(y_te20, p1, b_t1, idx1)
results.append({'Method': 'Method 1: 80/20 Split (Test Tuned)', 'Test Rows': len(X_te20), 'Threshold': round(b_t1, 2), 'Accuracy': accuracy_score(y_te20, pred1), 'Recall': recall_score(y_te20, pred1), 'Total_Loss': l1, 'Loss_Per_Order': l1/len(X_te20)})

# --- Method 2: 80/20 Split + 5-Fold OOF CV ---
skf = StratifiedKFold(n_splits=5, shuffle=False)
oof_p = np.zeros(len(X_tr80))
for tr_i, va_i in skf.split(X_tr80, y_tr80):
    spw = (y_tr80.iloc[tr_i] == 0).sum() / (y_tr80.iloc[tr_i] == 1).sum()
    m = XGBClassifier(n_estimators=300, max_depth=8, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, scale_pos_weight=spw, reg_alpha=0.1, reg_lambda=1.0, random_state=42, eval_metric='logloss')
    m.fit(X_tr80.iloc[tr_i], y_tr80.iloc[tr_i], sample_weight=w_tr80.iloc[tr_i] if w_tr80 is not None else None)
    oof_p[va_i] = m.predict_proba(X_tr80.iloc[va_i])[:, 1]
b_t2, m_l2 = 0.5, float('inf')
for t in thresholds:
    l, _, _, _ = compute_loss(y_tr80, oof_p, t, X_tr80.index)
    if l < m_l2: m_l2, b_t2 = l, t
l2, fn2, fp2, pred2 = compute_loss(y_te20, p1, b_t2, idx1)
results.append({'Method': 'Method 2: 80/20 Split (5-Fold OOF CV)', 'Test Rows': len(X_te20), 'Threshold': round(b_t2, 2), 'Accuracy': accuracy_score(y_te20, pred2), 'Recall': recall_score(y_te20, pred2), 'Total_Loss': l2, 'Loss_Per_Order': l2/len(X_te20)})

# --- Method 3: 70/15/15 Split ---
X_tmp3, X_te3, y_tmp3, y_te3, w_tmp3, w_te3 = train_test_split(X, y, sample_weights, test_size=0.15, random_state=42, shuffle=False)
X_tr3, X_va3, y_tr3, y_va3, w_tr3, w_va3 = train_test_split(X_tmp3, y_tmp3, w_tmp3, test_size=0.17647, random_state=42, shuffle=False)
spw3 = (y_tr3 == 0).sum() / (y_tr3 == 1).sum()
xgb3 = XGBClassifier(n_estimators=300, max_depth=8, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, scale_pos_weight=spw3, reg_alpha=0.1, reg_lambda=1.0, random_state=42, eval_metric='logloss')
xgb3.fit(X_tr3, y_tr3, sample_weight=w_tr3)
vp3 = xgb3.predict_proba(X_va3)[:, 1]
b_t3, m_l3 = 0.5, float('inf')
for t in thresholds:
    l, _, _, _ = compute_loss(y_va3, vp3, t, X_va3.index)
    if l < m_l3: m_l3, b_t3 = l, t
p3 = xgb3.predict_proba(X_te3)[:, 1]
l3, fn3, fp3, pred3 = compute_loss(y_te3, p3, b_t3, X_te3.index)
results.append({'Method': 'Method 3: 70/15/15 Split', 'Test Rows': len(X_te3), 'Threshold': round(b_t3, 2), 'Accuracy': accuracy_score(y_te3, pred3), 'Recall': recall_score(y_te3, pred3), 'Total_Loss': l3, 'Loss_Per_Order': l3/len(X_te3)})

# --- Method 4: 60/20/20 Split ---
X_tmp4, X_te4, y_tmp4, y_te4, w_tmp4, w_te4 = train_test_split(X, y, sample_weights, test_size=0.20, random_state=42, shuffle=False)
X_tr4, X_va4, y_tr4, y_va4, w_tr4, w_va4 = train_test_split(X_tmp4, y_tmp4, w_tmp4, test_size=0.25, random_state=42, shuffle=False)
spw4 = (y_tr4 == 0).sum() / (y_tr4 == 1).sum()
xgb4 = XGBClassifier(n_estimators=300, max_depth=8, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, scale_pos_weight=spw4, reg_alpha=0.1, reg_lambda=1.0, random_state=42, eval_metric='logloss')
xgb4.fit(X_tr4, y_tr4, sample_weight=w_tr4)
vp4 = xgb4.predict_proba(X_va4)[:, 1]
b_t4, m_l4 = 0.5, float('inf')
for t in thresholds:
    l, _, _, _ = compute_loss(y_va4, vp4, t, X_va4.index)
    if l < m_l4: m_l4, b_t4 = l, t
p4 = xgb4.predict_proba(X_te4)[:, 1]
l4, fn4, fp4, pred4 = compute_loss(y_te4, p4, b_t4, X_te4.index)
results.append({'Method': 'Method 4: 60/20/20 Split', 'Test Rows': len(X_te4), 'Threshold': round(b_t4, 2), 'Accuracy': accuracy_score(y_te4, pred4), 'Recall': recall_score(y_te4, pred4), 'Total_Loss': l4, 'Loss_Per_Order': l4/len(X_te4)})

res_df = pd.DataFrame(results)
res_df['Accuracy'] = res_df['Accuracy'].apply(lambda x: f'{x*100:.2f}%')
res_df['Recall'] = res_df['Recall'].apply(lambda x: f'{x*100:.2f}%')
res_df['Total_Loss'] = res_df['Total_Loss'].apply(lambda x: f'R$ {x:,.2f}')
res_df['Loss_Per_Order'] = res_df['Loss_Per_Order'].apply(lambda x: f'R$ {x:.4f}')

print('\n' + '='*80)
print('DATA SPLIT ABLATION STUDY RESULTS')
print('='*80)
print(res_df.to_string(index=False))
res_df.to_csv('split_experiment_results.csv', index=False)
m2_loss = res_df.loc[res_df['Method'].str.contains('Method 2'), 'Loss_Per_Order'].values[0]
print(f'\n✅ Experiment Complete: Method 2 (80/20 + 5-Fold OOF CV) selected with lowest per-order loss {m2_loss}.')


STEP 6.5: DATA SPLIT OPTIMIZATION EXPERIMENT & ABLATION STUDY
Dataset shape: 110,739 rows x 59 columns

DATA SPLIT ABLATION STUDY RESULTS
                               Method  Test Rows  Threshold Accuracy Recall    Total_Loss Loss_Per_Order
   Method 1: 80/20 Split (Test Tuned)      22148       0.76   90.65% 36.07% R$ 102,091.54      R$ 4.6095
Method 2: 80/20 Split (5-Fold OOF CV)      22148       0.61   89.49% 41.05% R$ 103,602.45      R$ 4.6777
             Method 3: 70/15/15 Split      16611       0.82   90.93% 34.38%  R$ 76,059.73      R$ 4.5789
             Method 4: 60/20/20 Split      22148       0.73   89.95% 38.28% R$ 102,769.34      R$ 4.6401

✅ Experiment Complete: Method 2 (80/20 + 5-Fold OOF CV) selected with lowest per-order loss R$ 4.6777.
